In [ ]:
import sys
from pathlib import Path

ROOT = Path('/content/drive/MyDrive/Logika Ui')
if not ROOT.exists():
    from google.colab import drive
    drive.mount('/content/drive')

ISI_CONFIG = '''from pathlib import Path

ROOT = Path('/content/drive/MyDrive/Logika Ui')
AKAR = ROOT / 'KTI_KARHUTLA'
HASIL = AKAR / 'hasil'
LAIN = AKAR / 'lainnya'

GADM_SHP = ROOT / 'GADM 2' / 'gadm41_IDN_2.shp'
GAMBUT_GEOJSON = LAIN / 'Indonesia_Peat_Lands.geojson'

TABEL_UTAMA = HASIL / 'tabel_gabungan_nasional_semua_bulan.csv'
PROB_FINAL = HASIL / 'prob_final_nasional.npz'
GAMBUT_KAB = HASIL / 'gambut_kabkota.csv'
VALIDASI_SIPONGI = HASIL / 'validasi_sipongi_firms_kabtahun.csv'
FIRMS_GADM = AKAR / 'firms_gadm'
FIRMS_BERSIH = HASIL / 'firms_nasional_harian_bersih.csv'
PRIORITAS_BERSIH = HASIL / 'prioritas_harian_bersih.csv'
HASIL_BACKTEST = HASIL / 'backtest_skor_harian.csv'
PRIORITAS_HARIAN = HASIL / 'prioritas_harian.csv'

PRIORITAS = ['Riau', 'Jambi', 'Sumatera Selatan',
             'Kalimantan Barat', 'Kalimantan Tengah', 'Kalimantan Selatan']
CRS_LUAS = 'EPSG:6933'
TAU = 0.05
'''
(ROOT / 'config_ui.py').write_text(ISI_CONFIG)
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import geopandas as gpd
import config_ui as cfg

Mounted at /content/drive


## 2 · Muat data

`p_K1` adalah peluang kelas K1 dari model C (ERA5 pukul 00 UTC, sebelum jendela hujan). Model D (dengan awan serentak) tidak dipakai di skor.
`api_7h` adalah jumlah titik panas 7 hari **sebelum** hari tersebut (`shift(1)`), jadi tidak memakai informasi hari yang sama.

In [ ]:
kolom = ['tanggal', 'GID_2', 'NAME_1', 'tahun', 'bulan', 'titik_panas']
tabel = pd.read_csv(cfg.TABEL_UTAMA, parse_dates=['tanggal'], usecols=kolom)

prob = np.load(cfg.PROB_FINAL)
assert len(prob['PC']) == len(tabel), 'urutan baris tidak sama dengan tabel'
tabel['p_K1'] = prob['PC'][:, 1]

tabel = tabel.sort_values(['GID_2', 'tanggal']).reset_index(drop=True)
tabel['api_7h'] = (tabel.groupby('GID_2')['titik_panas']
                   .transform(lambda s: s.shift(1).rolling(7, min_periods=4).sum()))

print(tabel.shape, '|', tabel['GID_2'].nunique(), 'wilayah |',
      tabel['tanggal'].min().date(), 'sampai', tabel['tanggal'].max().date())

(1187441, 8) | 479 wilayah | 2020-01-01 sampai 2026-10-14


In [ ]:
def hitung_gambut(ulang=False):
    if cfg.GAMBUT_KAB.exists() and not ulang:
        return pd.read_csv(cfg.GAMBUT_KAB)

    kab = gpd.read_file(cfg.GADM_SHP)[['GID_2', 'geometry']].to_crs(cfg.CRS_LUAS)
    kab['geometry'] = kab.geometry.make_valid()
    kab = kab.dissolve(by='GID_2').reset_index()

    gambut = gpd.read_file(cfg.GAMBUT_GEOJSON).to_crs(cfg.CRS_LUAS)
    gambut['geometry'] = gambut.geometry.make_valid()

    irisan = gpd.overlay(kab, gambut[['geometry']], how='intersection', keep_geom_type=True)
    luas_gambut = irisan.dissolve(by='GID_2').geometry.area / 1e4

    hasil = pd.DataFrame({'GID_2': kab['GID_2'].values,
                          'luas_kab_ha': kab.geometry.area.values / 1e4})
    hasil['luas_gambut_ha'] = hasil['GID_2'].map(luas_gambut).fillna(0.0)
    hasil['persen_gambut'] = 100 * hasil['luas_gambut_ha'] / hasil['luas_kab_ha']
    hasil.to_csv(cfg.GAMBUT_KAB, index=False)
    return hasil


gambut = hitung_gambut()
print(len(gambut), 'kabupaten/kota | total gambut:', round(gambut['luas_gambut_ha'].sum() / 1e6, 2), 'juta ha')
print('kabupaten/kota dengan gambut > 0:', int((gambut['luas_gambut_ha'] > 0).sum()))
print(gambut.sort_values('persen_gambut', ascending=False).head(8).round(2).to_string(index=False))

502 kabupaten/kota | total gambut: 14.96 juta ha
kabupaten/kota dengan gambut > 0: 128
      GID_2  luas_kab_ha  luas_gambut_ha  persen_gambut
 IDN.24.6_1    364109.52       332108.39          91.21
 IDN.24.2_1    227011.94       163010.80          71.81
IDN.14.12_1    972237.80       656249.23          67.50
 IDN.24.3_1   1342517.59       891677.32          66.42
 IDN.24.1_1    850887.20       532884.21          62.63
 IDN.12.6_1    859329.29       527009.30          61.33
IDN.24.12_1    782108.51       433004.04          55.36
 IDN.24.9_1   1306680.38       695922.72          53.26


In [ ]:
tabel = tabel.merge(gambut[['GID_2', 'persen_gambut']], on='GID_2', how='left')
tabel['persen_gambut'] = tabel['persen_gambut'].fillna(0.0)

per_kab = tabel.drop_duplicates('GID_2').set_index('GID_2')['persen_gambut']
tabel['r_gambut'] = tabel['GID_2'].map(per_kab.rank(pct=True))

tabel['r_awan'] = tabel.groupby('tanggal')['p_K1'].rank(pct=True)
tabel['r_api'] = tabel.groupby('tanggal')['api_7h'].rank(pct=True)
tabel['skor_tanpa_gambut'] = tabel['r_awan'] * tabel['r_api']
tabel['skor'] = tabel['skor_tanpa_gambut'] * tabel['r_gambut']
tabel['enam_prov'] = tabel['NAME_1'].isin(cfg.PRIORITAS)

simpan = ['tanggal', 'GID_2', 'NAME_1', 'p_K1', 'api_7h', 'persen_gambut',
          'skor_tanpa_gambut', 'skor', 'enam_prov']
tabel[simpan].to_csv(cfg.PRIORITAS_HARIAN, index=False, float_format='%.4f')
print('tersimpan:', cfg.PRIORITAS_HARIAN.name)

tersimpan: prioritas_harian.csv


In [ ]:
hari = tabel.loc[tabel['api_7h'].notna(), 'tanggal'].max()
h = tabel[tabel['tanggal'] == hari].sort_values('skor', ascending=False)
tampil = ['GID_2', 'NAME_1', 'p_K1', 'api_7h', 'persen_gambut', 'skor_tanpa_gambut', 'skor']

print('tanggal:', hari.date())
print('\nNASIONAL')
print(h[tampil].head(10).round(3).to_string(index=False))
print('\nENAM PROVINSI PRIORITAS')
print(h[h['enam_prov']][tampil].head(10).round(3).to_string(index=False))

tanggal: 2026-10-14

NASIONAL
      GID_2            NAME_1  p_K1  api_7h  persen_gambut  skor_tanpa_gambut  skor
IDN.22.10_1       Papua Barat 0.359   483.0         21.351              0.866 0.821
 IDN.22.7_1       Papua Barat 0.336   268.0         46.004              0.814 0.797
 IDN.22.1_1       Papua Barat 0.321  1023.0          6.036              0.832 0.711
 IDN.22.2_1       Papua Barat 0.282   842.0          6.591              0.738 0.638
 IDN.8.10_1             Jambi 0.233   515.0         39.149              0.653 0.635
 IDN.14.6_1 Kalimantan Tengah 0.220  2506.0         24.085              0.656 0.625
 IDN.23.4_1             Papua 0.373    57.0          3.138              0.762 0.623
  IDN.8.6_1             Jambi 0.225   343.0         44.898              0.633 0.619
IDN.12.11_1  Kalimantan Barat 0.383    23.0          6.569              0.684 0.590
 IDN.14.8_1 Kalimantan Tengah 0.159  1779.0         20.530              0.608 0.575

ENAM PROVINSI PRIORITAS
      GID_2          

In [ ]:
hari_uji = np.sort(tabel.loc[tabel['api_7h'].notna(), 'tanggal'].unique())[-90:]
dasar = (tabel[tabel['tanggal'].isin(hari_uji)]
         .pivot(index='tanggal', columns='GID_2', values='skor_tanpa_gambut')
         .fillna(0.0))
kab_urut = dasar.columns.to_numpy()
B = dasar.to_numpy()
g0 = per_kab.reindex(kab_urut).to_numpy()


def bobot(g):
    return pd.Series(g).rank(pct=True).to_numpy()


def top_k(S, k=10):
    return np.argpartition(-S, k, axis=1)[:, :k]


def irisan_top(Sa, Sb, k=10):
    a, b = top_k(Sa, k), top_k(Sb, k)
    return float(np.mean([len(set(x) & set(y)) / k for x, y in zip(a, b)]))


w0 = bobot(g0)
S0 = B * w0
rng = np.random.default_rng(1)

baris = []
for lmaks in (0.10, 0.25, 0.50):
    irisan, korelasi = [], []
    for _ in range(200):
        g1 = g0 * (1 - rng.uniform(0, lmaks, size=len(g0)))
        w1 = bobot(g1)
        irisan.append(irisan_top(S0, B * w1))
        korelasi.append(np.corrcoef(w0, w1)[0, 1])
    baris.append({'kehilangan_maks_per_kab': lmaks,
                  'irisan_top10_rata2': np.mean(irisan),
                  'irisan_top10_persentil5': np.percentile(irisan, 5),
                  'korelasi_peringkat_gambut': np.mean(korelasi)})
print(pd.DataFrame(baris).round(3).to_string(index=False))

 kehilangan_maks_per_kab  irisan_top10_rata2  irisan_top10_persentil5  korelasi_peringkat_gambut
                    0.10               0.992                    0.987                        1.0
                    0.25               0.978                    0.971                        1.0
                    0.50               0.956                    0.944                        1.0


## 7 · Batas atas gambut yang mungkin terdampak 2020 sampai 2025

Satu-satunya "perkiraan lahan hilang" yang bisa dipertanggungjawabkan adalah **batas atas**: anggap seluruh luas karhutla SiPongi (semua jenis lahan, dijumlah 2020 sampai 2025, lahan yang terbakar ulang terhitung berkali-kali) jatuh di gambut dan gambutnya habis. Hasilnya batas yang sangat longgar, bukan estimasi. Lalu uji apakah peringkat berubah di skenario ekstrem itu.

In [ ]:
sip = pd.read_csv(cfg.VALIDASI_SIPONGI)
terbakar = sip[sip['tahun'].between(2020, 2025)].groupby('GID_2')['luas_ha'].sum()

gam = gambut.set_index('GID_2')
batas = (terbakar.reindex(gam.index).fillna(0.0)
         / gam['luas_gambut_ha'].where(gam['luas_gambut_ha'] > 0)).clip(upper=1.0).fillna(0.0)

ada_gambut = gam['luas_gambut_ha'] > 0
print('kabupaten/kota dengan gambut:', int(ada_gambut.sum()))
print(batas[ada_gambut].describe(percentiles=[.5, .75, .9]).round(3).to_string())
print('batas atas > 10%:', int((batas[ada_gambut] > 0.10).sum()),
      '| > 50%:', int((batas[ada_gambut] > 0.50).sum()))

g_ekstrem = g0 * (1 - batas.reindex(kab_urut).fillna(0.0).to_numpy())
w_ekstrem = bobot(g_ekstrem)
print('\nirisan top-10 skenario ekstrem vs peta 2019 (rata-rata 90 hari):',
      round(irisan_top(S0, B * w_ekstrem), 3))
print('korelasi peringkat gambut:', round(np.corrcoef(w0, w_ekstrem)[0, 1], 3))

kabupaten/kota dengan gambut: 128
count    128.000
mean       0.229
std        0.336
min        0.000
50%        0.060
75%        0.303
90%        1.000
max        1.000
batas atas > 10%: 52 | > 50%: 23

irisan top-10 skenario ekstrem vs peta 2019 (rata-rata 90 hari): 0.957
korelasi peringkat gambut: 0.949


In [ ]:
def rasio_jul_jun(d):
    rata = d.groupby(['tahun', 'bulan'])['titik_panas'].mean().unstack('bulan')
    return (rata[7] / rata[6]).round(2)


hasil_rasio = pd.DataFrame({
    'seluruh Indonesia': rasio_jul_jun(tabel),
    'enam provinsi prioritas': rasio_jul_jun(tabel[tabel['enam_prov']]),
})
print(hasil_rasio.to_string())

       seluruh Indonesia  enam provinsi prioritas
tahun                                            
2020                1.80                     1.21
2021                2.55                     3.60
2022                2.47                     4.03
2023                2.13                     3.78
2024                2.86                     4.46
2025                4.29                     8.57
2026                4.23                     6.61
